# Temperature, Pressure, Runs, and Daytime Strikeouts at Coors Field

**Goals:**

- Examine how runs co-move with **temperature quintiles** at Coors Field.
- Examine how **strikeouts** vary across **pressure quintiles**.
- Investigate more deeply why **strikeouts increase during the day** while **runs stay relatively consistent**.

The deeper diagnostic sections focus on whether daytime strikeout increases are offset by other run-scoring channels such as hits, home runs, barrels, exit velocity, or better run conversion on balls in play.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# Load league-wide game data and filter to COL home games
data = pd.read_csv('../../data/league_weather_2021_2025.csv')
col = data[data['home_team'] == 'COL'].copy()

col['game_date'] = pd.to_datetime(col['game_date'])
col['month'] = col['game_date'].dt.month
month_map = {3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun', 7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct'}
col['month_name'] = col['month'].map(month_map)

col['temp_bin'] = pd.qcut(col['temp_f'], q=5, duplicates='drop')
col['pres_bin'] = pd.qcut(col['pres'], q=5, duplicates='drop')
col['time_of_day'] = col['start_hour'].apply(lambda h: 'Day' if h < 17 else 'Night')

# Derived diagnostics
col['strikeouts_per_100_pitches'] = 100 * col['strikeouts'] / col['total_pitches']
col['runs_per_hit'] = col['total_runs'] / col['hits']

print(f'Total COL home games: {len(col)}')
print('\nTemperature bins and game counts:')
print(col['temp_bin'].value_counts().sort_index())
print('\nPressure bins and game counts:')
print(col['pres_bin'].value_counts().sort_index())
print('\nDay vs. night counts:')
print(col['time_of_day'].value_counts())

In [ ]:
# ============================================================
# Temperature Quintiles: Summary Table
# ============================================================
temp_summary = col.groupby('temp_bin', observed=True).agg(
    games=('temp_f', 'size'),
    temp_mean=('temp_f', 'mean'),
    temp_std=('temp_f', 'std'),
    total_runs_mean=('total_runs', 'mean'),
    total_runs_std=('total_runs', 'std'),
    home_runs_mean=('home_runs_scored', 'mean'),
    away_runs_mean=('away_runs_scored', 'mean'),
    strikeouts_mean=('strikeouts', 'mean'),
    pres_mean=('pres', 'mean'),
    rhum_mean=('rhum', 'mean'),
    wspd_mean=('wspd_mph', 'mean'),
).round(2)

temp_summary.index.name = 'Temperature Quintile (degF)'
temp_summary.columns = [
    'Games', 'Temp Mean', 'Temp Std', 'Total Runs Mean', 'Total Runs Std',
    'Home Runs Mean', 'Away Runs Mean', 'Strikeouts Mean',
    'Pressure Mean', 'Humidity Mean', 'Wind Mean'
]
temp_summary

In [ ]:
# ============================================================
# Runs and Weather Profile by Temperature Quintile
# ============================================================
bin_order = col['temp_bin'].cat.categories
tick_labels = [str(b).replace('(', '').replace(']', '').replace(', ', ' to ') for b in bin_order]
x = np.arange(len(bin_order))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

grouped_runs = col.groupby('temp_bin', observed=True)['total_runs']
means = grouped_runs.mean().reindex(bin_order)
sems = grouped_runs.sem().reindex(bin_order)
axes[0, 0].bar(x, means, yerr=sems, capsize=4, color='#d62728', alpha=0.8, edgecolor='black', linewidth=0.5)
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(tick_labels, rotation=20, ha='right')
axes[0, 0].set_ylabel('Total Runs')
axes[0, 0].set_title('Total Runs by Temperature Quintile')
axes[0, 0].yaxis.grid(True, alpha=0.3)

bar_width = 0.35
home_means = col.groupby('temp_bin', observed=True)['home_runs_scored'].mean().reindex(bin_order)
away_means = col.groupby('temp_bin', observed=True)['away_runs_scored'].mean().reindex(bin_order)
axes[0, 1].bar(x - bar_width / 2, home_means, width=bar_width, color='#1f77b4', alpha=0.8, label='Home Runs Scored', edgecolor='black', linewidth=0.5)
axes[0, 1].bar(x + bar_width / 2, away_means, width=bar_width, color='#ff7f0e', alpha=0.8, label='Away Runs Scored', edgecolor='black', linewidth=0.5)
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(tick_labels, rotation=20, ha='right')
axes[0, 1].set_ylabel('Runs')
axes[0, 1].set_title('Home vs Away Runs by Temperature Quintile')
axes[0, 1].legend()
axes[0, 1].yaxis.grid(True, alpha=0.3)

axes[1, 0].plot(x, col.groupby('temp_bin', observed=True)['pres'].mean().reindex(bin_order), marker='o', color='#9467bd', linewidth=2)
axes[1, 0].plot(x, col.groupby('temp_bin', observed=True)['rhum'].mean().reindex(bin_order), marker='o', color='#1f77b4', linewidth=2)
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(tick_labels, rotation=20, ha='right')
axes[1, 0].set_title('Pressure and Humidity by Temperature Quintile')
axes[1, 0].set_ylabel('Pressure / Humidity')
axes[1, 0].legend(['Pressure', 'Humidity'])
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(x, col.groupby('temp_bin', observed=True)['wspd_mph'].mean().reindex(bin_order), marker='o', color='#2ca02c', linewidth=2)
axes[1, 1].plot(x, col.groupby('temp_bin', observed=True)['strikeouts'].mean().reindex(bin_order), marker='o', color='#8c564b', linewidth=2)
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(tick_labels, rotation=20, ha='right')
axes[1, 1].set_title('Wind and Strikeouts by Temperature Quintile')
axes[1, 1].set_ylabel('Wind / Strikeouts')
axes[1, 1].legend(['Wind Speed (mph)', 'Strikeouts'])
axes[1, 1].grid(True, alpha=0.3)

fig.suptitle('Coors Field: Runs and Co-Moving Conditions Across Temperature Quintiles', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Pressure Quintiles: Summary Table
# ============================================================
pres_summary = col.groupby('pres_bin', observed=True).agg(
    games=('pres', 'size'),
    pres_mean=('pres', 'mean'),
    pres_std=('pres', 'std'),
    strikeouts_mean=('strikeouts', 'mean'),
    strikeouts_std=('strikeouts', 'std'),
    total_runs_mean=('total_runs', 'mean'),
    temp_mean=('temp_f', 'mean'),
    rhum_mean=('rhum', 'mean'),
    wspd_mean=('wspd_mph', 'mean'),
).round(2)

pres_summary.index.name = 'Pressure Quintile (hPa)'
pres_summary.columns = [
    'Games', 'Pressure Mean', 'Pressure Std', 'Strikeouts Mean', 'Strikeouts Std',
    'Total Runs Mean', 'Temp Mean', 'Humidity Mean', 'Wind Mean'
]
pres_summary

In [ ]:
# ============================================================
# Strikeouts and Weather Profile by Pressure Quintile
# ============================================================
bin_order = col['pres_bin'].cat.categories
tick_labels = [str(b).replace('(', '').replace(']', '').replace(', ', ' to ') for b in bin_order]
x = np.arange(len(bin_order))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

grouped_k = col.groupby('pres_bin', observed=True)['strikeouts']
means = grouped_k.mean().reindex(bin_order)
sems = grouped_k.sem().reindex(bin_order)
axes[0, 0].bar(x, means, yerr=sems, capsize=4, color='#1f77b4', alpha=0.8, edgecolor='black', linewidth=0.5)
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(tick_labels, rotation=20, ha='right')
axes[0, 0].set_ylabel('Strikeouts')
axes[0, 0].set_title('Strikeouts by Pressure Quintile')
axes[0, 0].yaxis.grid(True, alpha=0.3)

axes[0, 1].bar(x, col.groupby('pres_bin', observed=True)['total_runs'].mean().reindex(bin_order), color='#d62728', alpha=0.8, edgecolor='black', linewidth=0.5)
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(tick_labels, rotation=20, ha='right')
axes[0, 1].set_ylabel('Total Runs')
axes[0, 1].set_title('Total Runs by Pressure Quintile')
axes[0, 1].yaxis.grid(True, alpha=0.3)

axes[1, 0].plot(x, col.groupby('pres_bin', observed=True)['temp_f'].mean().reindex(bin_order), marker='o', color='#d62728', linewidth=2)
axes[1, 0].plot(x, col.groupby('pres_bin', observed=True)['rhum'].mean().reindex(bin_order), marker='o', color='#1f77b4', linewidth=2)
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(tick_labels, rotation=20, ha='right')
axes[1, 0].set_title('Temperature and Humidity by Pressure Quintile')
axes[1, 0].set_ylabel('Temp / Humidity')
axes[1, 0].legend(['Temperature', 'Humidity'])
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(x, col.groupby('pres_bin', observed=True)['wspd_mph'].mean().reindex(bin_order), marker='o', color='#2ca02c', linewidth=2)
axes[1, 1].plot(x, col.groupby('pres_bin', observed=True)['away_runs_scored'].mean().reindex(bin_order), marker='o', color='#ff7f0e', linewidth=2)
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(tick_labels, rotation=20, ha='right')
axes[1, 1].set_title('Wind and Away Runs by Pressure Quintile')
axes[1, 1].set_ylabel('Wind / Away Runs')
axes[1, 1].legend(['Wind Speed (mph)', 'Away Runs'])
axes[1, 1].grid(True, alpha=0.3)

fig.suptitle('Coors Field: Strikeouts and Co-Moving Conditions Across Pressure Quintiles', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Day vs. Night: Expanded Summary Table
# ============================================================
day_night_summary = col.groupby('time_of_day').agg(
    games=('strikeouts', 'size'),
    strikeouts_mean=('strikeouts', 'mean'),
    strikeouts_std=('strikeouts', 'std'),
    strikeouts_per_100_pitches_mean=('strikeouts_per_100_pitches', 'mean'),
    total_runs_mean=('total_runs', 'mean'),
    total_runs_std=('total_runs', 'std'),
    hits_mean=('hits', 'mean'),
    walks_mean=('walks', 'mean'),
    home_runs_hit_mean=('home_runs_hit', 'mean'),
    avg_exit_velocity_mean=('avg_exit_velocity', 'mean'),
    barrel_rate_mean=('barrel_rate', 'mean'),
    n_barrels_mean=('n_barrels', 'mean'),
    hr_h_ratio_mean=('hr_h_ratio', 'mean'),
    runs_per_hit_mean=('runs_per_hit', 'mean'),
    total_pitches_mean=('total_pitches', 'mean'),
    temp_mean=('temp_f', 'mean'),
    pres_mean=('pres', 'mean'),
    rhum_mean=('rhum', 'mean'),
    wspd_mean=('wspd_mph', 'mean'),
).round(3)

day_night_summary.columns = [
    'Games', 'Strikeouts Mean', 'Strikeouts Std', 'K per 100 Pitches',
    'Total Runs Mean', 'Total Runs Std', 'Hits Mean', 'Walks Mean',
    'Home Runs Hit Mean', 'Avg Exit Velo', 'Barrel Rate', 'Barrels Mean',
    'HR:H Ratio', 'Runs per Hit', 'Total Pitches Mean',
    'Temp Mean', 'Pressure Mean', 'Humidity Mean', 'Wind Mean'
]
day_night_summary

In [ ]:
# ============================================================
# Day vs. Night: Offense Compensation Diagnostics
# ============================================================
time_order = ['Day', 'Night']
x = np.arange(len(time_order))
colors = ['#ff9900', '#003366']

metrics = [
    ('strikeouts', 'Strikeouts'),
    ('strikeouts_per_100_pitches', 'K per 100 Pitches'),
    ('hits', 'Hits'),
    ('walks', 'Walks'),
    ('home_runs_hit', 'Home Runs Hit'),
    ('avg_exit_velocity', 'Avg Exit Velocity'),
    ('barrel_rate', 'Barrel Rate'),
    ('total_runs', 'Total Runs'),
]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, (col_name, title) in zip(axes.flat, metrics):
    means = col.groupby('time_of_day')[col_name].mean().reindex(time_order)
    ax.bar(x, means, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(time_order)
    ax.set_title(title)
    ax.yaxis.grid(True, alpha=0.3)

fig.suptitle('If Daytime Strikeouts Rise, What Offsets Them?', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Day vs. Night Within the Same Weather Buckets
# ============================================================
temp_order = col['temp_bin'].cat.categories
pres_order = col['pres_bin'].cat.categories
temp_labels = [str(b).replace('(', '').replace(']', '').replace(', ', ' to ') for b in temp_order]
pres_labels = [str(b).replace('(', '').replace(']', '').replace(', ', ' to ') for b in pres_order]
x_temp = np.arange(len(temp_order))
x_pres = np.arange(len(pres_order))
width = 0.35

fig, axes = plt.subplots(2, 2, figsize=(18, 10))

for i, label in enumerate(time_order):
    subset = col[col['time_of_day'] == label]
    color = colors[i]
    offset = (i - 0.5) * width

    temp_k = subset.groupby('temp_bin', observed=True)['strikeouts'].mean().reindex(temp_order)
    pres_k = subset.groupby('pres_bin', observed=True)['strikeouts'].mean().reindex(pres_order)
    temp_runs = subset.groupby('temp_bin', observed=True)['total_runs'].mean().reindex(temp_order)
    pres_runs = subset.groupby('pres_bin', observed=True)['total_runs'].mean().reindex(pres_order)

    axes[0, 0].bar(x_temp + offset, temp_k, width, label=label, color=color, alpha=0.8, edgecolor='black', linewidth=0.5)
    axes[0, 1].bar(x_pres + offset, pres_k, width, label=label, color=color, alpha=0.8, edgecolor='black', linewidth=0.5)
    axes[1, 0].bar(x_temp + offset, temp_runs, width, label=label, color=color, alpha=0.8, edgecolor='black', linewidth=0.5)
    axes[1, 1].bar(x_pres + offset, pres_runs, width, label=label, color=color, alpha=0.8, edgecolor='black', linewidth=0.5)

axes[0, 0].set_title('Strikeouts by Temperature Quintile and Time of Day')
axes[0, 0].set_ylabel('Strikeouts')
axes[0, 0].set_xticks(x_temp)
axes[0, 0].set_xticklabels(temp_labels, rotation=20, ha='right')
axes[0, 0].legend()
axes[0, 0].yaxis.grid(True, alpha=0.3)

axes[0, 1].set_title('Strikeouts by Pressure Quintile and Time of Day')
axes[0, 1].set_ylabel('Strikeouts')
axes[0, 1].set_xticks(x_pres)
axes[0, 1].set_xticklabels(pres_labels, rotation=20, ha='right')
axes[0, 1].legend()
axes[0, 1].yaxis.grid(True, alpha=0.3)

axes[1, 0].set_title('Total Runs by Temperature Quintile and Time of Day')
axes[1, 0].set_ylabel('Total Runs')
axes[1, 0].set_xticks(x_temp)
axes[1, 0].set_xticklabels(temp_labels, rotation=20, ha='right')
axes[1, 0].legend()
axes[1, 0].yaxis.grid(True, alpha=0.3)

axes[1, 1].set_title('Total Runs by Pressure Quintile and Time of Day')
axes[1, 1].set_ylabel('Total Runs')
axes[1, 1].set_xticks(x_pres)
axes[1, 1].set_xticklabels(pres_labels, rotation=20, ha='right')
axes[1, 1].legend()
axes[1, 1].yaxis.grid(True, alpha=0.3)

fig.suptitle('Does the Daytime Strikeout Gap Persist Within the Same Temperature and Pressure Ranges?', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Day vs. Night by Month: Strikeouts, Runs, and Pressure
# ============================================================
months = sorted(col['month'].dropna().unique())
month_names = [month_map.get(m, str(m)) for m in months]
x = np.arange(len(months))
width = 0.35

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, label in enumerate(time_order):
    subset = col[col['time_of_day'] == label]
    grouped_k = subset.groupby('month')['strikeouts'].mean().reindex(months)
    grouped_runs = subset.groupby('month')['total_runs'].mean().reindex(months)
    grouped_p = subset.groupby('month')['pres'].mean().reindex(months)
    offset = (i - 0.5) * width
    color = colors[i]
    axes[0].bar(x + offset, grouped_k, width, label=label, color=color, alpha=0.8, edgecolor='black', linewidth=0.5)
    axes[1].bar(x + offset, grouped_runs, width, label=label, color=color, alpha=0.8, edgecolor='black', linewidth=0.5)
    axes[2].bar(x + offset, grouped_p, width, label=label, color=color, alpha=0.8, edgecolor='black', linewidth=0.5)

axes[0].set_xticks(x)
axes[0].set_xticklabels(month_names)
axes[0].set_ylabel('Strikeouts')
axes[0].set_title('Monthly Strikeouts: Day vs Night')
axes[0].legend()
axes[0].yaxis.grid(True, alpha=0.3)

axes[1].set_xticks(x)
axes[1].set_xticklabels(month_names)
axes[1].set_ylabel('Total Runs')
axes[1].set_title('Monthly Total Runs: Day vs Night')
axes[1].legend()
axes[1].yaxis.grid(True, alpha=0.3)

axes[2].set_xticks(x)
axes[2].set_xticklabels(month_names)
axes[2].set_ylabel('Pressure (hPa)')
axes[2].set_title('Monthly Pressure: Day vs Night')
axes[2].legend()
axes[2].yaxis.grid(True, alpha=0.3)

fig.suptitle('Do Daytime Strikeouts Track Pressure Without Changing the Run Environment by Month?', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()